# Personalized Hotel Ranking Engine

Four-track ranking project with top-k recommendation inference and monitoring hooks.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import load_expedia_data, prepare_candidate_scoring_table, query_time_split, build_reference_baselines
from src.features import build_preprocessor, prepare_model_inputs
from src.ranking import (
    run_lazypredict_discovery,
    select_top3_eligible_families,
    run_manual_engineering_track,
    run_flaml_track,
    run_pycaret_track,
    build_topk_recommendations,
    save_inference_bundle,
)
from src.evaluation import build_leaderboard

SEED = 42
PROJECT_NAME = 'personalized-hotel-ranking-engine'
PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'expedia'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Business Problem and Success Criteria

Return top-5 hotel recommendations per search with strong booking relevance.

Primary metric: MAP@5
Secondary: HitRate@5
Tertiary: inference latency

## 2) Dataset Access and Data Dictionary

Dataset: Expedia Hotel Recommendations (`train.csv`).

The project reformulates logs into supervised candidate scoring rows.

In [ ]:
data = load_expedia_data(RAW_DIR, sample_frac=0.005, random_state=SEED)
data['train'].shape

## 3) Data Cleaning and Leakage Checks

- Construct label from booking signal
- Group-safe negative sampling
- Remove leakage-prone post-event fields

In [ ]:
candidate_df = prepare_candidate_scoring_table(data['train'], negative_ratio=5, random_state=SEED)
candidate_df[['label']].value_counts()

## 4) Feature Engineering

- Context and hotel candidate attributes
- Date-derived search signals
- Categorical encoding and numeric scaling

In [ ]:
train_df, holdout_df = query_time_split(candidate_df, holdout_frac=0.2, random_state=SEED)
preprocessor = build_preprocessor(train_df.drop(columns=['label', 'srch_id', 'date_time'], errors='ignore'))
X_train, X_holdout, y_train, y_holdout, holdout_meta = prepare_model_inputs(
    train_df=train_df,
    holdout_df=holdout_df,
    preprocessor=preprocessor,
    target_col='label',
    group_col='srch_id',
    item_col='hotel_cluster',
)
X_train.shape, X_holdout.shape

## 5) Validation Strategy

- Query/time-aware holdout split
- Ranking evaluation at search-id level
- Business metrics centered on top-5 usefulness

In [ ]:
summary = pd.DataFrame({
    'split': ['train', 'holdout'],
    'rows': [len(train_df), len(holdout_df)],
    'positive_rate': [train_df['label'].mean(), holdout_df['label'].mean()]
})
summary

## 6) Track 1: LazyPredict Discovery Lab

Run candidate scoring discovery on encoded matrix.

In [ ]:
lazy_table = run_lazypredict_discovery(X_train, X_holdout, y_train, y_holdout)
lazy_table.head(15)

## 7) Selection of Top 3 Eligible Models

Only top-3 eligible families from LazyPredict proceed to manual lab.

In [ ]:
eligible_table, top3_families = select_top3_eligible_families(
    lazy_table=lazy_table,
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    holdout_meta=holdout_meta,
    random_state=SEED,
)
eligible_table, top3_families

## 8) Track 2: Manual Engineering Lab

Manual top-3 candidate scoring with explicit score-to-top-k conversion.

In [ ]:
manual_results, manual_models, manual_scored = run_manual_engineering_track(
    top3_families=top3_families,
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    holdout_meta=holdout_meta,
    random_state=SEED,
)
manual_results[['model_name', 'map_at_5', 'hit_rate_at_5', 'p95_latency_ms']]

## 9) Track 3: FLAML Optimization Lab

Budget-aware challenger search with ranking-centric evaluation.

In [ ]:
flaml_result = run_flaml_track(
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    holdout_meta=holdout_meta,
    time_budget=120,
    random_state=SEED,
)
flaml_result

## 10) Track 4: PyCaret Experiment Lab

PyCaret candidate-scoring experiment evaluated with ranking metrics.

In [ ]:
pycaret_train = train_df.drop(columns=['date_time'], errors='ignore').copy()
pycaret_holdout = holdout_df.drop(columns=['date_time'], errors='ignore').copy()
pycaret_result = run_pycaret_track(
    train_df=pycaret_train,
    holdout_df=pycaret_holdout,
    target_col='label',
    session_id=SEED,
    model_output_path=ARTIFACT_DIR / 'pycaret_expedia_ranking',
)
pycaret_result

## 11) Unified Leaderboard and Final Model Ranking

Includes popularity/recent-booking baselines plus all 4 tracks.

In [ ]:
baseline_rows, holdout_with_baselines = build_reference_baselines(train_df, holdout_df)
leaderboard = build_leaderboard(
    project_name=PROJECT_NAME,
    baseline_rows=baseline_rows,
    lazy_results=eligible_table,
    manual_results=manual_results,
    flaml_result=flaml_result,
    pycaret_result=pycaret_result,
)
leaderboard.head(10)

In [ ]:
leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_expedia_ranking.csv', index=False)
leaderboard[['model_name', 'library_source', 'holdout_primary_metric', 'holdout_secondary_metric', 'final_rank']].head(10)

## 12) Business Recommendation

After run completion, explain winner, safer second-best scenario, and excluded tradeoff.

## 13) Inference / Deployment Path

Generate top-5 recommendations from winning scorer and save sample output.

In [ ]:
winner = leaderboard.sort_values('final_rank').iloc[0]
if winner['library_source'] == 'manual' and winner['model_name'] in manual_scored:
    scored_df = manual_scored[winner['model_name']]
    if winner['model_name'] in manual_models:
        save_inference_bundle(manual_models[winner['model_name']], preprocessor, ARTIFACT_DIR)
elif winner['library_source'] == 'flaml':
    scored_df = flaml_result.get('scored', pd.DataFrame())
elif winner['library_source'] == 'pycaret':
    scored_df = pycaret_result.get('scored', pd.DataFrame())
elif winner['model_name'] == 'popularity_baseline':
    scored_df = holdout_with_baselines[['srch_id', 'hotel_cluster', 'label', 'score_popularity']].rename(columns={'score_popularity': 'score'})
else:
    scored_df = holdout_with_baselines[['srch_id', 'hotel_cluster', 'label', 'score_recent_booking']].rename(columns={'score_recent_booking': 'score'})

top5 = build_topk_recommendations(scored_df, k=5)
top5.to_csv(ARTIFACT_DIR / 'top5_sample_recommendations.csv', index=False)
top5.head()

## 14) Monitoring / Drift / Retraining Plan

Track MAP@5 and HitRate@5 over sliding windows, candidate coverage drift, and cold-start performance by segment.

## 15) Limitations and Next Steps

- Add richer user history features
- Add explicit diversity constraints in top-k output
- Add counterfactual policy evaluation before full rollout